# 01 Environment and Data

This notebook documents the simulation environment and the data used to calibrate the Formula 1 reinforcement learning experiment.

It demonstrates:
- project imports from `src/f1_rl_safety`
- the expected project folder structure
- basic checks for available data files
- a lightweight environment smoke test
- inspection of race data used for calibration

In [1]:
from pathlib import Path
import sys
import os
import json

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

print("Project root:", ROOT)
print("src exists:", (ROOT / "src").exists())
print("data exists:", (ROOT / "data").exists())
print("models exists:", (ROOT / "models").exists())
print("logs exists:", (ROOT / "logs").exists())

Project root: /Users/joel_mathew/f1-rl-safety
src exists: True
data exists: True
models exists: True
logs exists: True


## Import project modules

These imports come from the main source code used in the project.
If an import fails, check that:
1. the notebook is inside the project folder,
2. `src/` exists,
3. the package path is `src/f1_rl_safety/`.

In [3]:
from f1_rl_safety.f1_env import F1RaceEnv
from f1_rl_safety.data_loader import *

## Inspect project structure

This gives a quick view of the key folders that support reproducibility.

In [4]:
key_paths = [
    ROOT / "src" / "f1_rl_safety",
    ROOT / "data",
    ROOT / "logs",
    ROOT / "models",
    ROOT / "archive",
]

for p in key_paths:
    print(f"\n{p}")
    if p.exists():
        for child in sorted(p.iterdir()):
            print("  -", child.name)
    else:
        print("  [missing]")


/Users/joel_mathew/f1-rl-safety/src/f1_rl_safety
  - .DS_Store
  - .venv
  - __init__.py
  - __pycache__
  - data_loader.py
  - eval_policies.py
  - f1_env.py
  - train.py

/Users/joel_mathew/f1-rl-safety/data
  - cache
  - silverstone_2025_laps.csv

/Users/joel_mathew/f1-rl-safety/logs
  - rulebook
  - safe
  - unconstrained

/Users/joel_mathew/f1-rl-safety/models
  - ppo_rulebook.zip
  - ppo_safe.zip
  - ppo_unconstrained.zip

/Users/joel_mathew/f1-rl-safety/archive
  - logs_old_20260613_204626
  - logs_old_20260613_210650
  - models_old_20260613_204626
  - models_old_20260613_210650


## Locate likely race data files

This searches the `data/` directory for CSV, pickle and cache files that may have been used during calibration.

In [5]:
data_dir = ROOT / "data"
candidate_files = []

if data_dir.exists():
    for ext in ["*.csv", "*.pkl", "*.ff1pkl", "*.json"]:
        candidate_files.extend(data_dir.rglob(ext))

print(f"Found {len(candidate_files)} candidate data files.")
for f in candidate_files[:50]:
    print("-", f.relative_to(ROOT))

Found 12 candidate data files.
- data/silverstone_2025_laps.csv
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/car_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/weather_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/driver_info.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/position_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/lap_count.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/session_status_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/race_control_messages.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/timing_app_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/track_status_data.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/session_info.ff1pkl
- data/cache/2025/2025-07-06_British_Grand_Prix/2025-07-06_Race/_extended_timin

## Load tabular race data if available

If you have a processed CSV such as `silverstone_2025_laps.csv`, this cell will display it.
If you only have FastF1 cache files, that is still acceptable for submission, but a CSV export is easier for assessors to inspect.

In [6]:
import pandas as pd

csv_files = sorted(data_dir.rglob("*.csv")) if data_dir.exists() else []
if csv_files:
    selected_csv = csv_files[0]
    print("Using:", selected_csv.relative_to(ROOT))
    df = pd.read_csv(selected_csv)
    display(df.head())
    print(df.columns.tolist())
    print("Rows:", len(df))
else:
    print("No CSV files found under data/.")

Using: data/silverstone_2025_laps.csv


,Driver,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound,TyreLife,FreshTyre,Stint,TrackStatus,IsAccurate
0,VER,8.0,0 days 00:01:45.834000,0 days 00:00:32.223000,0 days 00:00:43.021000,0 days 00:00:30.590000,INTERMEDIATE,8.0,True,1.0,1,True
1,VER,9.0,0 days 00:01:46.326000,0 days 00:00:32.688000,0 days 00:00:43.578000,0 days 00:00:30.060000,INTERMEDIATE,9.0,True,1.0,1,True
2,VER,10.0,0 days 00:01:46.432000,0 days 00:00:32.657000,0 days 00:00:44.139000,0 days 00:00:29.636000,INTERMEDIATE,10.0,True,1.0,1,True
3,VER,13.0,0 days 00:01:51.224000,0 days 00:00:33.824000,0 days 00:00:47.545000,0 days 00:00:29.855000,INTERMEDIATE,2.0,True,2.0,12,True
4,VER,22.0,0 days 00:01:51.018000,0 days 00:00:36.073000,0 days 00:00:45.732000,0 days 00:00:29.213000,INTERMEDIATE,11.0,True,2.0,12,True


['Driver', 'LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Compound', 'TyreLife', 'FreshTyre', 'Stint', 'TrackStatus', 'IsAccurate']
Rows: 496


## Environment smoke test

This cell performs a minimal environment check:
- create the environment
- reset once
- sample one action
- step once

This is not a full experiment run; it simply confirms that the environment is callable.

In [7]:
import numpy as np

env = F1RaceEnv()
obs, info = env.reset()

print("Reset successful.")
print("Observation type:", type(obs))
print("Info type:", type(info))
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)

action = env.action_space.sample()
step_output = env.step(action)

print("\nStep output length:", len(step_output))
print("Sample action:", action)
print("Sample reward:", step_output[1])
print("Terminated:", step_output[2], "Truncated:", step_output[3])

Reset successful.
Observation type: <class 'numpy.ndarray'>
Info type: <class 'dict'>
Action space: Box([ 0.  0. -1.], [1. 4. 1.], (3,), float32)
Observation space: Box(-1.0, 1.0, (18,), float32)

Step output length: 5
Sample action: [0.72872925 1.9068686  0.09325797]
Sample reward: -1.1404672262014468
Terminated: False Truncated: False


## Notes

This notebook is included to help readers understand:
- how the custom environment is organised,
- what data files are present,
- and whether the environment can be instantiated successfully.

The main implementation remains in:
- `src/f1_rl_safety/f1_env.py`
- `src/f1_rl_safety/data_loader.py`